# Лаба 03. Чат-агент на LangChain: память диалога, свой компонент, развилки

Продолжение лабы 02: тот же наставник с памятью фактов; новое — в разделе «Доработка» ниже и в трёх последних разделах.

Тема семестрового проекта — **проверка Go-кода по код-стайлу команды**. Здесь это наставник, который знакомится
с новым разработчиком: по одному задаёт вопросы (имя, уровень, опыт, стек, проект, слабые места, цель),
запоминает **факты** в `memory.json` и после перезапуска продолжает с тех вопросов, на которые ответа ещё нет.
Когда всё известно, объясняет правила команды S1–S6 с учётом уровня и слабых мест.

Как устроено:

- **Чат** — цепочка `ChatPromptTemplate | ChatOpenAI`, обёрнутая в `RunnableWithMessageHistory`.
  История сообщений живёт только в RAM текущего сеанса.
- **Память** — `FileFactMemory(BaseChatMemory)`: `load_memory_variables()` отдаёт известные факты и список
  недостающих, `save_context()` извлекает факты из реплики **отдельным LLM-вызовом со structured output**
  (`with_structured_output(UserFacts, method="json_schema")`) и пишет `memory.json`.
  Файл есть — читаем, нет — пустая память. Ключ `user_id` — несколько профилей в одном файле.
- **Проверка фактов.** 3B-модель заполняет поля «на всякий случай» (имя «junior», проект «Docker»), поэтому
  извлечённый факт записывается, только если он есть в реплике (`accept_fact`), а уже известные факты
  не перезаписываются. Отброшенное видно в служебном логе.
- **Служебная информация** (что прочитано, извлечено, записано, какой вопрос дальше) выводится в отдельную
  ячейку «Служебный лог» через `display_id`, а не в чат.

Доработка (лаба 03):

- **Память диалога** — `DialogHistory(BaseChatMessageHistory)` для `RunnableWithMessageHistory`: в промпт идёт
  окно последних сообщений (`trim_messages`). Режим `buffer` — остальное отбрасывается, `summary` — вытесненные
  сообщения сворачиваются отдельным LLM-вызовом в резюме, оно идёт в системный промпт.
- **Свой компонент** — `TurnParser(Runnable)`: разбирает реплику и ответ LLM-классификатора (`Route`: класс,
  правило, настроение) в параметры хода; явные признаки важнее ответа 3B-модели.
  Действия: инструмент `check_code` (`@tool`, линтер S1–S6 на регулярках) и элемент памяти `progress`
  в `memory.json` (уровень по квизу, ошибки по правилам).
- **Развилки** — `RunnableBranch`: по классу вопроса (объяснить правило / проверить код / квиз / другое);
  по недовольству и повтору просьбы — «упростить»; по уровню знаний из памяти — глубина объяснения и сложность
  вопросов квиза (два верных ответа подряд — уровень выше, две ошибки — ниже).

Модель — Qwen2.5-3B-Instruct Q4_K_M в llama-server (OpenAI-совместимый API). Для `json_schema`
llama-server строит грамматику по схеме, поэтому даже 3B-модель всегда отдаёт валидный JSON.

```bash
pip install -r requirements.txt
llama-server -m qwen2.5-3b-instruct-q4_k_m.gguf -c 8192
```

In [1]:
import os

# OpenAI-совместимый сервер: llama-server (локально или на Kaggle) или OpenRouter
os.environ.setdefault("LLM_BASE_URL", "http://127.0.0.1:8080/v1")  # OpenRouter: https://openrouter.ai/api/v1
os.environ.setdefault("LLM_MODEL", "local")                         # OpenRouter: имя модели, ключ в LLM_API_KEY

MEMORY_FILE = "memory.json"
USER_ID = "artur"  # ключ профиля в memory.json

Код агента сохраняется в модуль, чтобы после перезапуска ядра его можно было просто импортировать заново.

In [2]:
%%writefile fact_agent.py
"""Чат-наставник по Go-код-стайлу команды с долгосрочной памятью фактов в файле.

Память — не история сообщений, а набор фактов о пользователе (имя, уровень, стек, цели...).
Факты извлекаются отдельным LLM-вызовом со structured output и хранятся в memory.json по user_id.
История сообщений живёт только в RAM текущего сеанса и после перезапуска пропадает; в промпт она
попадает не целиком: окно последних сообщений (buffer) или окно плюс резюме вытесненных (summary).

Когда факты собраны — режим наставника: классификатор реплики → TurnParser (свой Runnable: разбирает
параметры и решает, куда идти) → RunnableBranch: объяснить правило, проверить код инструментом check_code,
квиз по уровню знаний (прогресс — в memory.json), упростить объяснение, если пользователь недоволен.
"""

import json
import os
import re
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Literal, Optional

from IPython.display import Markdown, display
from langchain_classic.memory.chat_memory import BaseChatMemory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import AIMessage, BaseMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core._api import LangChainDeprecationWarning
from pydantic import BaseModel, Field, create_model

# предупреждения об устаревших классах мешаются с выводом чата
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

STYLE_GUIDE = """\
S1. Имена — MixedCaps/camelCase, без подчёркиваний. Аббревиатуры целиком: userID, не userId.
S2. Ошибки не игнорировать: ни `_ = f()`, ни вызов без проверки возвращаемой ошибки. `defer x.Close()` допускается.
S3. В HTTP-обработчиках не использовать panic — вернуть 500 через http.Error.
S4. Функции, которые ходят в БД или сеть, принимают ctx context.Context первым параметром.
    HTTP-обработчики берут контекст из r.Context() — отдельный параметр ctx им не нужен.
S5. Ошибки, возвращаемые наверх, оборачиваются: fmt.Errorf("контекст: %w", err).
S6. У экспортируемых функций и методов есть doc-комментарий, начинающийся с имени."""
RULES = {rule: " ".join(text.split())
         for rule, text in re.findall(r"^(S\d)\. (.+?)(?=^S\d\.|\Z)", STYLE_GUIDE, re.S | re.M)}
RULE_WORDS = {  # о каком правиле речь, если номер не назван
    "S1": ("имен", "camel", "подчёрк", "подчерк", "аббревиатур"),
    "S2": ("игнор", "не проверя"),
    "S3": ("panic", "паник"),
    "S4": ("context", "ctx", "контекст"),
    "S5": ("оборач", "errorf", "%w", "ошибк"),
    "S6": ("коммент",),
}
RULE_NUMBER = r"(?<!\w)[SsСс]\s?([1-6])(?!\d)"  # S3, s3, С3 (кириллица)


def find_rule(text: str) -> Optional[str]:
    if m := re.search(RULE_NUMBER, text):
        return f"S{m.group(1)}"
    low = text.lower()
    return next((rule for rule, words in RULE_WORDS.items() if any(w in low for w in words)), None)


def lint_go(code: str) -> list[tuple[int, str, str]]:
    """Простой линтер по правилам S1–S6 на регулярках: (строка, правило, что не так)."""
    found = []
    lines = code.splitlines()
    handler = "http.ResponseWriter" in code
    for n, line in enumerate(lines, 1):
        s = line.strip()
        if s.startswith("//"):
            continue
        names = re.findall(r"\b(?:func\s+(?:\([^)]*\)\s*)?|var\s+|const\s+)([A-Za-z_]\w*)", s)
        if m := re.match(r"(?:if\s+|for\s+)?([\w\s,]+?)\s*:=", s):
            names += [x.strip() for x in m.group(1).split(",")]
        for name in names:
            if "_" in name.strip("_"):
                found.append((n, "S1", f"имя `{name}` с подчёркиванием"))
            elif re.search(r"(Id|Url|Http|Json|Sql)(?=[A-Z]|$)", name):
                found.append((n, "S1", f"аббревиатура в `{name}` — целиком заглавными"))
        if re.search(r"(?:^|[\s(])_\s*(?:,\s*_\s*)?=", s) or re.search(r",\s*_\s*:?=", s):
            found.append((n, "S2", "ошибка присвоена `_`"))
        elif re.fullmatch(r"[\w.]+(?:\([^)]*\))?\.(?:Encode|Decode|Write|Exec|Scan)\(.*\)", s):
            found.append((n, "S2", "результат-ошибка вызова не проверяется"))
        if handler and re.search(r"\bpanic\(", s):
            found.append((n, "S3", "panic в HTTP-обработчике"))
        if m := re.search(r"\.(Query|QueryRow|Exec)\((?!\))", s):  # у url.Query() аргументов нет
            found.append((n, "S4", f"запрос к БД без контекста: нужен {m.group(1)}Context(ctx, ...)"))
        if re.fullmatch(r"return\s+(?:[^,]+,\s*)*err", s):
            found.append((n, "S5", "ошибка возвращается без обёртки fmt.Errorf(\"...: %w\", err)"))
        elif "fmt.Errorf(" in s and "err" in s and "%w" not in s:
            found.append((n, "S5", "в fmt.Errorf ошибка через %v, а не %w"))
        if m := re.match(r"func\s+(?:\([^)]*\)\s*)?([A-Z]\w*)", s):
            prev = lines[n - 2].strip() if n > 1 else ""
            if not prev.startswith(f"// {m.group(1)} "):
                found.append((n, "S6", f"у экспортируемой `{m.group(1)}` нет doc-комментария, начинающегося с имени"))
    return found


@tool
def check_code(code: str) -> str:
    """Проверяет Go-код по правилам команды S1–S6 и возвращает нарушения с номерами строк."""
    return "\n".join(f"строка {n}: {rule} — {what}" for n, rule, what in lint_go(code)) or "нарушений не найдено"


# Квиз: в каждом фрагменте одно нарушение, ответ — номер правила. Проверяет ответ код, объяснение — why:
# пересказывая ответ, 3B-модель противоречила вердикту («Неверно: S5. Верно: S2…»)
QUIZ = [
    {"id": "j1", "level": "junior", "rule": "S1",
     "code": "func load_config(path string) (*Config, error) {",
     "why": "`load_config` — имя с подчёркиванием, по правилам `loadConfig`."},
    {"id": "j2", "level": "junior", "rule": "S2",
     "code": "data, _ := os.ReadFile(path)",
     "why": "ошибка из `os.ReadFile` присвоена `_`: нужно `data, err := os.ReadFile(path)` и проверить `err`."},
    {"id": "j3", "level": "junior", "rule": "S3",
     "code": "func ordersHandler(w http.ResponseWriter, r *http.Request) {\n"
             "    orders, err := repo.List(r.Context())\n    if err != nil {\n        panic(err)\n    }",
     "why": "panic в HTTP-обработчике роняет обработку запроса; нужно `http.Error(w, \"internal error\", http.StatusInternalServerError)` и `return`."},
    {"id": "m1", "level": "middle", "rule": "S4",
     "code": "// FindUser ищет пользователя по id.\nfunc (r *Repo) FindUser(id int64) (*User, error) {\n"
             "    row := r.db.QueryRow(\"SELECT name FROM users WHERE id = $1\", id)",
     "why": "`FindUser` ходит в БД, но не принимает контекст: нужно `FindUser(ctx context.Context, id int64)` и `QueryRowContext(ctx, ...)`."},
    {"id": "m2", "level": "middle", "rule": "S5",
     "code": "if err := row.Scan(&u.Name); err != nil {\n    return nil, err\n}",
     "why": "ошибка проверена, но уходит наверх без контекста: нужно `return nil, fmt.Errorf(\"scan user: %w\", err)`. S2 — это когда ошибку не проверяют вовсе, а здесь её проверяют."},
    {"id": "m3", "level": "middle", "rule": "S6",
     "code": "func ParseOrder(b []byte) (*Order, error) {",
     "why": "у экспортируемой `ParseOrder` нет doc-комментария; нужен `// ParseOrder разбирает заказ из JSON.`"},
    {"id": "s1", "level": "senior", "rule": "S1",
     "code": "// GetUserId возвращает id текущего пользователя.\n"
             "func (s *Service) GetUserId(ctx context.Context) (int64, error) {",
     "why": "аббревиатура пишется целиком: `GetUserID`, а не `GetUserId`."},
    {"id": "s2", "level": "senior", "rule": "S5",
     "code": "return nil, fmt.Errorf(\"load order %d: %v\", id, err)",
     "why": "с `%v` ошибка превращается в текст и `errors.Is`/`errors.As` перестают её видеть; нужен `%w`."},
    {"id": "s3", "level": "senior", "rule": "S6",
     "code": "// возвращает заказы клиента\nfunc ListOrders(ctx context.Context, clientID int64) ([]Order, error) {",
     "why": "комментарий есть, но не начинается с имени функции: нужно `// ListOrders возвращает заказы клиента.`"},
    {"id": "s4", "level": "senior", "rule": "S2",
     "code": "func statsHandler(w http.ResponseWriter, r *http.Request) {\n"
             "    json.NewEncoder(w).Encode(collectStats())\n}",
     "why": "`Encode` возвращает ошибку, а её не проверяют: `if err := json.NewEncoder(w).Encode(...); err != nil { http.Error(...) }`."},
]
LEVELS = ["junior", "middle", "senior"]
DEPTH = {
    "junior": "объясняй простыми словами, без лишних терминов; обязательно короткий пример кода «плохо / хорошо»",
    "middle": "объясни правило и зачем оно нужно, дай короткий пример «плохо / хорошо»",
    "senior": "коротко: суть правила, неочевидные случаи и исключения; пример — только если он показывает тонкость",
}


class UserFacts(BaseModel):
    """Факты о разработчике. Поле заполняется, только если пользователь сам это сказал."""

    name: Optional[str] = Field(None, description="Имя пользователя")
    # Literal превращается в enum схемы: грамматика llama-server не даст записать сюда «год»
    go_level: Optional[Literal["junior", "middle", "senior"]] = Field(None, description="Уровень в Go")
    go_experience: Optional[str] = Field(None, description="Сколько времени пишет на Go: «1 год», «полгода»")
    stack: Optional[str] = Field(None, description="Технологии кроме Go: базы данных, инструменты, ОС")
    project: Optional[str] = Field(None, description="Над каким проектом или сервисом работает")
    weak_spots: Optional[str] = Field(None, description="Что даётся трудно в Go или в код-стайле")
    goal: Optional[str] = Field(None, description="Чего хочет добиться")


# порядок опроса и дословные вопросы: 3B-модель, получив «узнай уровень», спрашивает что-то своё
QUESTIONS = {
    "name": "Как тебя зовут?",
    "go_level": "Какой у тебя уровень в Go: junior, middle или senior?",
    "go_experience": "Сколько ты уже пишешь на Go?",
    "stack": "С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?",
    "project": "Над каким проектом ты сейчас работаешь?",
    "weak_spots": "Что тебе сейчас даётся труднее всего в Go или в код-стайле?",
    "goal": "Чего ты хочешь добиться?",
}
LABELS = {
    "name": "Имя", "go_level": "Уровень", "go_experience": "Опыт в Go", "stack": "Стек",
    "project": "Проект", "weak_spots": "Слабые места", "goal": "Цель",
}
INCIDENTAL = ("name", "go_level", "go_experience")  # можно назвать, отвечая на другой вопрос
EMPTY = {"", "null", "none", "нет", "неизвестно", "не указано", "n/a"}


def make_llm(temperature: float = 0.3) -> ChatOpenAI:
    """Любой OpenAI-совместимый сервер: llama-server локально/на Kaggle или OpenRouter."""
    return ChatOpenAI(
        base_url=os.environ.get("LLM_BASE_URL", "http://127.0.0.1:8080/v1"),
        api_key=os.environ.get("LLM_API_KEY", "none"),
        model=os.environ.get("LLM_MODEL", "local"),
        temperature=temperature,
        max_tokens=400,
    )


class LogCell:
    """Служебный вывод в собственной ячейке ноутбука, а не в чате.

    Ячейка, в которой создан LogCell, держит display_id; все записи обновляют её вывод.
    """

    def __init__(self, title: str = "Служебный лог"):
        self.title = title
        self.lines: list[str] = []
        self.handle = display(Markdown(self._render()), display_id=True)

    def __call__(self, text: str):
        self.lines.append(f"- `{datetime.now():%H:%M:%S}` {text}")
        self.handle.update(Markdown(self._render()))

    def _render(self) -> str:
        return f"**{self.title}**\n\n" + ("\n".join(self.lines) or "_пока пусто_")


def format_facts(facts: dict) -> str:
    if not facts:
        return "пока ничего не известно"
    return "\n".join(f"- {LABELS[k]}: {v}" for k, v in facts.items() if k in LABELS)


EXTRACT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты извлекаешь факты о пользователе из его последнего сообщения и возвращаешь JSON по схеме.\n"
     "Заполняй поле, только если пользователь сам прямо сказал это в своём сообщении. "
     "Всё, о чём он не сказал, — null. Не придумывай, не копируй слова из вопроса ассистента. "
     "Пиши значения коротко, по-русски."),
    ("human", "Ассистент спрашивал про поле «{field}».\nВопрос ассистента: {question}\n"
              "Сообщение пользователя: {message}"),
])


LEVEL_WORDS = {"junior": ("junior", "джун"), "middle": ("middle", "мидл"), "senior": ("senior", "сеньор", "синьор")}


def accept_fact(key: str, value: str, message: str, asked: str) -> bool:
    """Проверка, что факт действительно есть в реплике: маленькая модель заполняет поля «на всякий случай»."""
    text = message.lower()
    if key == "name":
        return value.lower() in text and (asked == "name" or "зовут" in text or re.search(r"\bя\s", text))
    if key == "go_level":
        return any(word in text for word in LEVEL_WORDS[value])
    if key == "go_experience":
        unit = r"\d|год|лет|месяц|полгода|недел"
        return bool(re.search(unit, value.lower())) and bool(re.search(unit, text))
    # стек, проект, слабые места, цель — только в ответ на свой вопрос
    return key == asked


class FileFactMemory(BaseChatMemory):
    """Долгосрочная память фактов в JSON-файле: {user_id: {"facts": {...}, "progress": {...}, "updated_at": ...}}.

    load_memory_variables() — отдаёт известные факты и список недостающих;
    save_context() — извлекает факты из реплики пользователя отдельным LLM-вызовом и пишет файл;
    progress — уровень знаний по квизу и ошибки по правилам (меняется в режиме наставника, save_progress()).
    """

    path: Path
    user_id: str
    llm: Any = None  # модель для извлечения фактов
    log: Any = None
    facts: dict = Field(default_factory=dict)
    progress: dict = Field(default_factory=dict)

    def model_post_init(self, __context: Any) -> None:
        profile = self._read_all().get(self.user_id)
        self.facts = dict(profile["facts"]) if profile else {}
        self.progress = dict(profile.get("progress", {})) if profile else {}
        if self.log:
            if not self.path.exists():
                self.log(f"`{self.path}` нет — память пустая")
            elif profile:
                self.log(f"прочитан `{self.path}`, профиль `{self.user_id}`: "
                         f"`{json.dumps(self.facts, ensure_ascii=False)}`")
                if self.progress:
                    self.log(f"прогресс: `{json.dumps(self.progress, ensure_ascii=False)}`")
            else:
                self.log(f"в `{self.path}` нет профиля `{self.user_id}` — память пустая")

    @property
    def memory_variables(self) -> list[str]:
        return ["facts", "missing"]

    def missing(self) -> list[str]:
        return [k for k in QUESTIONS if k not in self.facts]

    def load_memory_variables(self, inputs: dict[str, Any]) -> dict[str, Any]:
        return {"facts": format_facts(self.facts), "missing": self.missing()}

    def save_context(self, inputs: dict[str, Any], outputs: dict[str, str]) -> None:
        """inputs: {"input": реплика, "question": вопрос ассистента, "field": о каком факте он был}.

        Историю сообщений не сохраняем — только факты, прошедшие проверку.
        """
        if not self.missing():
            if self.log:
                self.log("все факты известны — извлечение не нужно")
            return
        message, field = inputs["input"], inputs.get("field") or "—"
        # В схеме — поле, о котором спросили, и недостающие короткие факты, которые называют попутно.
        # Иначе 3B-модель раскладывает перечисление по полям: stack="PostgreSQL", project="Docker"...
        fields = [k for k in self.missing() if k == field or k in INCIDENTAL]
        spec = {k: (UserFacts.model_fields[k].annotation, UserFacts.model_fields[k]) for k in fields}
        if field in fields and field not in INCIDENTAL:
            # Необязательное поле модель охотно оставляет null («Биллинг: считаем платежи...» → project=null),
            # поэтому спрошенное поле — обязательная строка с выходом «нет».
            spec[field] = (str, Field(description=UserFacts.model_fields[field].description
                                      + "; если пользователь об этом не сказал — «нет»"))
        schema = create_model("UserFacts", **spec)
        extractor = EXTRACT_PROMPT | self.llm.with_structured_output(schema, method="json_schema")
        result = extractor.invoke({"message": message, "question": inputs.get("question") or "—", "field": field})
        raw = {k: v.strip() for k, v in result.model_dump().items()
               if isinstance(v, str) and v.strip().lower() not in EMPTY}
        accepted, rejected = {}, {}
        for key, value in raw.items():
            if key in self.facts:
                rejected[key] = "уже известно"
            elif not accept_fact(key, value, message, field):
                rejected[key] = value
            else:
                accepted[key] = value
        if self.log:
            self.log(f"модель извлекла: `{json.dumps(raw, ensure_ascii=False)}`")
            if rejected:
                self.log(f"отброшено проверкой: `{json.dumps(rejected, ensure_ascii=False)}`")
        if accepted:
            self.facts.update(accepted)
            self._write()
            if self.log:
                self.log(f"записано в `{self.path}`: {', '.join(accepted)}")

    def level(self) -> str:
        """Уровень для сложности тем и вопросов: из квиза, а до первого квиза — названный пользователем."""
        return self.progress.get("level") or self.facts.get("go_level") or "junior"

    def save_progress(self, **changes) -> None:
        self.progress.update(changes)
        self._write()
        if self.log:
            self.log(f"прогресс записан в `{self.path}`: `{json.dumps(self.progress, ensure_ascii=False)}`")

    def clear(self) -> None:
        self.facts, self.progress = {}, {}
        self._write()

    def _read_all(self) -> dict:
        if not self.path.exists():
            return {}
        return json.loads(self.path.read_text(encoding="utf-8") or "{}")

    def _write(self) -> None:
        data = self._read_all()
        data[self.user_id] = {"facts": self.facts, "updated_at": datetime.now().isoformat(timespec="seconds")}
        if self.progress:
            data[self.user_id]["progress"] = self.progress
        self.path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")




SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Перескажи реплики разговора наставника по Go с разработчиком одним предложением, которое начинается "
     "со слов «Пользователь»: что он сообщил или о чём спросил. Сохрани имена людей и названия. "
     "Объяснения наставника и код не пересказывай. Только то, что есть в репликах, по-русски."),
    ("human", "{lines}"),
])


class DialogHistory(BaseChatMessageHistory):
    """История сообщений сеанса (только RAM); в промпт попадает не вся.

    buffer  — последние window сообщений, остальное просто отбрасывается;
    summary — те же window сообщений плюс резюме вытесненных: выпавшие из окна реплики отдельный LLM-вызов
              пересказывает одним-двумя предложениями, пункт добавляется в резюме (оно идёт в системный промпт).
              Пункты копятся, а не переписываются: переписывая резюме целиком, 3B-модель теряла старое.
    """

    def __init__(self, mode: str = "summary", window: int = 6, llm: Any = None, log: Any = None):
        assert mode in ("buffer", "summary")
        self.mode, self.window, self.llm = mode, window, llm
        self.log = log or (lambda text: None)
        self.full: list[BaseMessage] = []
        self.points: list[str] = []  # резюме — по пункту на свёрнутый кусок
        self.summarized = 0  # сколько первых сообщений уже свёрнуто в резюме

    @property
    def messages(self) -> list[BaseMessage]:
        # token_counter=len — считаем сообщения, а не токены
        return trim_messages(self.full, strategy="last", token_counter=len, max_tokens=self.window,
                             start_on="human")

    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.full.extend(messages)
        dropped = len(self.full) - len(self.messages)
        if self.mode == "summary" and dropped > self.summarized:
            chunk = self.full[self.summarized:dropped]
            # одно приветствие не сворачиваем: из него модель сочиняла целый диалог
            if any(m.type == "human" for m in chunk):
                # ответы наставника обрезаем: иначе 3B-модель копирует их в резюме вместо пересказа
                lines = "\n".join(f"Пользователь: {m.content}" if m.type == "human" else
                                   f"Наставник: {m.content[:150]}…" for m in chunk)
                lines = re.sub(r"```.*?(```|$)", "[код]", lines, flags=re.S)
                point = (SUMMARY_PROMPT | self.llm).invoke({"lines": lines}).content.strip()
                self.points.append(re.sub(r"^(новое )?резюме:\s*", "", point, flags=re.I))
                self.log(f"резюме: свёрнуто сообщений {len(chunk)} → «{self.points[-1]}»")
            self.summarized = dropped

    def prompt_summary(self) -> str:
        if self.mode == "summary" and self.points:
            return "Краткое содержание более раннего разговора:\n" + "\n".join(f"- {p}" for p in self.points) + "\n\n"
        return ""

    def clear(self) -> None:
        self.full, self.points, self.summarized = [], [], 0


CHAT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — наставник по код-стайлу Go в команде разработки. Отвечай только по-русски, без английских "
     "и китайских слов (кроме кода), на «ты», кратко: 2–4 предложения.\n\n"
     "Что ты знаешь о пользователе (долгосрочная память):\n{facts}\n\n"
     "{summary}Задача на этот ход: {task}"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])


class Route(BaseModel):
    """Класс реплики пользователя в режиме наставника."""

    intent: Literal["explain_rule", "review_code", "quiz", "other"] = Field(
        description="explain_rule — просит объяснить правило или тему код-стайла, показать пример; "
                    "review_code — прислал код на Go или просит проверить код; "
                    "quiz — просит проверить его знания, задать вопрос, устроить тест; "
                    "other — всё остальное")
    rule: Literal["S1", "S2", "S3", "S4", "S5", "S6", "нет"] = Field(
        description="О каком правиле речь; «нет», если ни о каком")
    # поле mood («недоволен») убрано: в первом прогоне модель ставила его почти на каждую реплику


ROUTE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Определи, чего хочет пользователь наставника по код-стайлу Go, и верни JSON по схеме.\n"
               f"Правила команды:\n{STYLE_GUIDE}"),
    ("human", "{input}"),
])

NEGATIVE = re.compile(r"не понял|не понимаю|непонятн|не ясно|запутал|слишком сложно|бред|ерунд|\bне то\b"
                      r"|вс[её] равно|опять", re.I)
QUIZ_WORDS = re.compile(r"квиз|тест|проверь меня|проверь мои знания|задай.{0,10}вопрос|ещё вопрос|еще вопрос", re.I)
MORE_WORDS = re.compile(r"^(ещё|еще|дальше|давай|следующий)\b", re.I)
EXPLAIN_WORDS = re.compile(r"объясн|расскаж|покажи|пример|что такое|зачем", re.I)
CODE_BLOCK = re.compile(r"```(?:go)?\s*\n(.*?)```", re.S)


def similar(a: str, b: str) -> bool:
    """Повтор той же просьбы: больше половины слов совпадает."""
    wa, wb = set(re.findall(r"\w+", a.lower())), set(re.findall(r"\w+", b.lower()))
    return len(wa) >= 3 and len(wa & wb) / len(wa | wb) >= 0.5


class TurnParser(Runnable[dict, dict]):
    """Свой компонент цепочки: разбирает реплику и ответ классификатора в параметры хода.

    На входе {"input", "route": Route, ...}, на выходе тот же словарь плюс intent, rule, code, reason.
    3B-модель ошибается в классе («Объясни правило S3» → review_code), поэтому явные признаки важнее её
    ответа: блок кода → проверка кода (без кода проверять нечего), ожидается ответ на вопрос квиза → проверка
    ответа, «не понял» или повтор просьбы → упростить, номер правила → объяснить.
    """

    def __init__(self, agent: "FactAgent"):
        self.agent = agent

    def invoke(self, input: dict, config=None, **kwargs) -> dict:
        agent, text, route = self.agent, input["input"], input["route"]
        intent, reasons = route.intent, []
        code = m.group(1) if (m := CODE_BLOCK.search(text)) else (
            text if "\n" in text and re.search(r"\bfunc\b|:=|\bif err\b", text) else "")
        if code:
            intent = "review_code"
        elif agent.pending_quiz and (re.search(r"(?<!\w)[SsСс]?\s?[1-6](?!\d)", text)
                                     or re.search(r"не знаю", text, re.I)):
            intent = "quiz_answer"
        elif QUIZ_WORDS.search(text) or (agent.last_intent in ("quiz", "quiz_answer") and MORE_WORDS.search(text)):
            intent = "quiz"
        elif intent == "review_code" or (intent == "other" and re.search(RULE_NUMBER, text)):
            intent = "explain_rule"
        if intent == "explain_rule" and not (find_rule(text) or EXPLAIN_WORDS.search(text)):
            intent = "other"  # «Кто у нас делает ревью?» — не про правило, хоть модель и назвала S1

        negative = bool(NEGATIVE.search(text))
        repeated = similar(text, agent.last_user)
        if negative:
            reasons.append("в реплике недовольство или непонимание")
        if repeated:
            reasons.append("повторяет прошлую просьбу")
        if reasons and intent in ("explain_rule", "other"):
            intent = "simplify"

        weak = find_rule(agent.memory.facts.get("weak_spots", ""))
        if intent == "simplify":  # упрощаем то, что объясняли, а не правило, названное в жалобе
            rule = agent.last_rule or find_rule(text) or weak
        else:
            rule = find_rule(text) or (route.rule if route.rule != "нет" else None)
            if intent == "other" and rule and EXPLAIN_WORDS.search(text):
                intent = "explain_rule"
            if intent == "explain_rule" and not rule:
                rule = weak or agent.last_rule or "S1"
        agent.frustration = agent.frustration + 1 if intent == "simplify" else 0
        if intent != "simplify":
            agent.quiz_context = None
        agent.last_intent = intent
        if intent in ("explain_rule", "simplify") and rule:
            agent.last_rule = rule
        agent.log(f"классификатор: `{route.model_dump_json()}` → парсер: intent=`{intent}`, rule=`{rule}`"
                  + (f", причина: {'; '.join(reasons)}" if reasons else ""))
        return {**input, "intent": intent, "rule": rule, "code": code, "reason": "; ".join(reasons)}


class FactAgent:
    def __init__(self, user_id: str, path: str | Path = "memory.json", log: LogCell | None = None,
                 dialog: str = "summary", window: int = 6):
        self.log = log or (lambda text: None)
        self.memory = FileFactMemory(
            path=Path(path), user_id=user_id, log=log, llm=make_llm(0))
        self.history = DialogHistory(dialog, window, llm=make_llm(0), log=self.log)
        self.chain = CHAT_PROMPT | make_llm(0.3)
        self.chat_chain = RunnableWithMessageHistory(
            self.chain, self._history, input_messages_key="input", history_messages_key="history")
        self.classifier = ROUTE_PROMPT | make_llm(0).with_structured_output(Route, method="json_schema")
        mentor = (
            RunnablePassthrough.assign(route=self.classifier)
            | TurnParser(self)
            | RunnableBranch(
                (lambda x: x["intent"] == "review_code", RunnableLambda(self._review)),
                (lambda x: x["intent"] == "quiz_answer", RunnableLambda(self._grade)),
                (lambda x: x["intent"] == "quiz", RunnableLambda(self._quiz)),
                (lambda x: x["intent"] == "simplify", RunnableLambda(self._simplify)),
                (lambda x: x["intent"] == "explain_rule", RunnableLambda(self._explain)),
                RunnableLambda(self._other),
            )
        )
        self.mentor_chain = RunnableWithMessageHistory(
            mentor, self._history, input_messages_key="input", history_messages_key="history")
        self.last_question = ""
        self.asked = ""  # о каком факте был последний вопрос ассистента
        # состояние режима наставника (RAM сеанса)
        self.pending_quiz: dict | None = None
        self.last_intent = self.last_rule = self.last_user = ""
        self.frustration = 0  # сколько ходов подряд пользователь недоволен
        self.quiz_context: dict | None = None  # неверный ответ на квиз — о чём упрощать
        self.log(f"память диалога: `{dialog}`, окно {window} сообщений")

    def _history(self, session_id: str) -> DialogHistory:
        return self.history

    def _task(self, start: bool) -> str:
        missing = self.memory.missing()
        if missing:
            ask = f"затем задай дословно этот вопрос и больше ничего не спрашивай: «{QUESTIONS[missing[0]]}»"
            if start and not self.memory.facts:
                return "поздоровайся и одной фразой представься; " + ask
            if start:
                return "пользователь вернулся: поприветствуй его по имени и одной фразой напомни, что уже знаешь; " + ask
            return "одной фразой отреагируй на ответ пользователя (если он что-то спросил — коротко ответь); " + ask
        offer = ("предложи на выбор: объяснить любое правило команды S1–S6, проверить его код "
                 "(пусть пришлёт фрагмент) или задать вопросы на знание правил")
        if start:
            weak = find_rule(self.memory.facts.get("weak_spots", ""))
            about = f" (к нему относится правило {weak}: «{RULES[weak]}»)" if weak else ""
            return ("пользователь вернулся, о нём известно всё нужное: поприветствуй по имени, "
                    f"напомни про его слабое место{about} и " + offer)
        return "поблагодари за ответы, одной фразой подытожь, что узнал о пользователе, и " + offer

    def _say(self, x: dict, task: str) -> AIMessage:
        return self.chain.invoke({**x, "task": task})

    def _note_mistakes(self, rules: list[str]) -> None:
        mistakes = dict(self.memory.progress.get("mistakes", {}))
        for rule in rules:
            mistakes[rule] = mistakes.get(rule, 0) + 1
        self.memory.save_progress(mistakes=mistakes)

    # --- ветки режима наставника ---

    def _explain(self, x: dict) -> AIMessage:
        level, rule = self.memory.level(), x["rule"]
        self.log(f"ветка «объяснить»: {rule}, глубина по уровню `{level}`")
        return self._say(x, f"объясни правило команды {rule}: «{RULES[rule]}». Уровень пользователя — {level}: "
                            f"{DEPTH[level]}. Пример кода (до 10 строк) — из его проекта.")

    def _review(self, x: dict) -> AIMessage:
        report = check_code.invoke({"code": x["code"]})  # инструмент: точный список нарушений
        rules = sorted({rule for _, rule, _ in lint_go(x["code"])})
        self.log(f"ветка «проверить код»: инструмент `check_code` → {', '.join(rules) or 'чисто'}")
        if rules:
            self._note_mistakes(rules)
            x["rule"] = rules[0]
        if not rules:
            return AIMessage(content="Инструмент check_code нарушений не нашёл — по правилам S1–S6 код чистый.")
        level = self.memory.level()
        reply = self._say(x, "пользователь прислал код. Инструмент check_code нашёл нарушения:\n"
                             f"{report}\nОбъясни каждое из них по одному предложению ({DEPTH[level]}) и покажи "
                             "исправленный код, где исправлены все они. Не называй нарушений, которых нет в списке.")
        # 3B-модель пересказывала отчёт не полностью (из S1, S4, S5 упомянула только S4) — список даём дословно
        return AIMessage(content=f"Инструмент check_code:\n{report}\n\n{reply.content.strip()}")

    def _quiz(self, x: dict) -> AIMessage:
        level, progress = self.memory.level(), self.memory.progress
        asked = progress.get("asked", [])
        candidates = [q for q in QUIZ if q["level"] == level and q["id"] not in asked]
        if not candidates:  # все вопросы уровня заданы — начинаем круг заново
            asked = [a for a in asked if not a.startswith(level[0])]
            candidates = [q for q in QUIZ if q["level"] == level]
        # сначала правила, на которых пользователь ошибался, и его слабое место
        weak, mistakes = find_rule(self.memory.facts.get("weak_spots", "")), progress.get("mistakes", {})
        q = max(candidates, key=lambda q: mistakes.get(q["rule"], 0) + (q["rule"] == weak))
        self.pending_quiz = q
        self.memory.save_progress(asked=asked + [q["id"]])
        self.log(f"ветка «квиз»: уровень `{level}`, вопрос `{q['id']}` (ответ {q['rule']})")
        return AIMessage(content=f"Вопрос уровня {level}. Какое правило (S1–S6) нарушено?\n```go\n{q['code']}\n```")

    def _grade(self, x: dict) -> AIMessage:
        q, self.pending_quiz = self.pending_quiz, None
        m = re.search(r"(?<!\w)[SsСс]?\s?([1-6])(?!\d)", x["input"])
        answer = f"S{m.group(1)}" if m else "не знаю"
        ok = answer == q["rule"]
        p = self.memory.progress
        level = self.memory.level()
        streak = max(p.get("streak", 0), 0) + 1 if ok else min(p.get("streak", 0), 0) - 1
        new_level = level
        if streak >= 2 and level != LEVELS[-1]:  # два верных подряд — вопросы сложнее
            new_level, streak = LEVELS[LEVELS.index(level) + 1], 0
        elif streak <= -2 and level != LEVELS[0]:  # две ошибки подряд — проще
            new_level, streak = LEVELS[LEVELS.index(level) - 1], 0
        mistakes = dict(p.get("mistakes", {}))
        if not ok:
            mistakes[q["rule"]] = mistakes.get(q["rule"], 0) + 1
        self.memory.save_progress(level=new_level, streak=streak, correct=p.get("correct", 0) + ok,
                                  wrong=p.get("wrong", 0) + (not ok), mistakes=mistakes)
        self.log(f"ветка «ответ на квиз»: ответ {answer}, верно {q['rule']} → {'верно' if ok else 'неверно'}"
                 + (f"; уровень `{level}` → `{new_level}`" if new_level != level else ""))
        self.last_rule = q["rule"]
        if not ok:
            self.quiz_context = {"q": q, "answer": answer}
        text = f"{'Верно' if ok else 'Неверно'}: здесь нарушено {q['rule']} — {q['why']}"
        if new_level != level:
            text += f" Дальше вопросы уровня {new_level}."
        return AIMessage(content=text)

    def _simplify(self, x: dict) -> AIMessage:
        rule = x["rule"]
        self.log(f"ветка «упростить»: {rule}, недоволен {self.frustration} ход(а) подряд")
        task = f"пользователь недоволен ({x['reason']}). Не оправдывайся. "
        if qc := self.quiz_context:
            task += (f"Он ответил на вопрос квиза {qc['answer']}, а верно {rule}. Код:\n{qc['q']['code']}\n"
                     f"Почему {rule}: {qc['q']['why']}\nПравило {rule}: «{RULES[rule]}». Объясни это ещё проще, "
                     f"своими словами, с бытовой аналогией; называй правило только как {rule}.")
        else:
            task += (f"Объясни правило {rule}: «{RULES[rule]}» совсем просто: одна бытовая аналогия и пример кода "
                     "3–6 строк «плохо / хорошо».")
        reply = self._say(x, task).content.strip()
        if self.frustration >= 2:  # просьбу в промпте 3B-модель пропускала — добавляем сами
            reply += "\n\nЕсли так и не стало понятнее — пришли свой фрагмент кода, разберём правило на нём."
        return AIMessage(content=reply)

    def _other(self, x: dict) -> AIMessage:
        self.log("ветка «другое»")
        return self._say(x, "если пользователь спрашивает о том, что говорил раньше, — ответь по истории и "
                            "краткому содержанию разговора, назови людей и детали оттуда. Если вопрос не про Go и не "
                            "про этот разговор — ты на него не отвечаешь и уточнений не просишь: одной фразой скажи, "
                            "что ты наставник только по код-стайлу Go, и предложи: объяснить правило, проверить код "
                            "или вопросы на знание правил.")

    # --- сеанс ---

    def start(self) -> str:
        """Первая реплика сеанса — без ввода пользователя."""
        self.log(f"сеанс начат: user_id=`{self.memory.user_id}`, pid процесса `{os.getpid()}`")
        self.log(f"не хватает фактов: {', '.join(self.memory.missing()) or 'нет'}")
        text = self.chain.invoke({"facts": format_facts(self.memory.facts), "task": self._task(start=True),
                                  "summary": "", "history": [], "input": "(пользователь открыл чат)"}
                                 ).content.strip()
        self.history.add_ai_message(text)
        self.last_question = text
        self.asked = next(iter(self.memory.missing()), "")
        return text

    def turn(self, user_text: str) -> str:
        config = {"configurable": {"session_id": self.memory.user_id}}
        if self.memory.missing() or self.asked:
            # знакомство: сначала факты — следующий вопрос должен учитывать только что сказанное
            self.memory.save_context(
                {"input": user_text, "question": self.last_question, "field": self.asked}, {})
            missing = self.memory.missing()
            self.log(f"дальше: {'спросить ' + missing[0] if missing else 'режим наставника — все факты собраны'}")
            variables = self.memory.load_memory_variables({})
            text = self.chat_chain.invoke(
                {"input": user_text, "facts": variables["facts"], "summary": self.history.prompt_summary(),
                 "task": self._task(start=False)}, config=config).content.strip()
            self.asked = next(iter(missing), "")
        else:
            self.log(f"история: в промпт {len(self.history.messages)} из {len(self.history.full)} сообщений"
                     + (", плюс резюме" if self.history.prompt_summary() else ""))
            reply = self.mentor_chain.invoke(
                {"input": user_text, "facts": format_facts(self.memory.facts),
                 "summary": self.history.prompt_summary()}, config=config)
            text = reply.content.strip()
            self.last_user = user_text
        self.last_question = text
        return text

    def chat(self, script: list[str] | None = None):
        """REPL. script — заранее заданные реплики (для демонстрации); без него читает input()."""
        print(f"ассистент> {self.start()}\n")
        replies = iter(script) if script is not None else None
        while True:
            if replies is None:
                user_text = input("вы> ").strip()
            else:
                user_text = next(replies, "/exit")
                print(f"вы> {user_text}")
            if user_text in ("/exit", "/quit", "выход"):
                self.log("сеанс завершён командой выхода")
                break
            if user_text:
                print(f"ассистент> {self.turn(user_text)}\n")


Writing fact_agent.py


## Сеанс 1 — первый запуск

Для воспроизводимой демонстрации реплики пользователя заданы списком; `agent.chat()` без аргументов читает ввод через `input()`.

In [3]:
from pathlib import Path

Path(MEMORY_FILE).unlink(missing_ok=True)  # чистый старт — только для демонстрации

import fact_agent as fa

**Служебный лог — отдельная ячейка.** Агент пишет сюда, пока идёт чат в ячейке ниже.

In [4]:
log = fa.LogCell("Служебный лог — сеанс 1")

**Служебный лог — сеанс 1**

- `05:16:56` `memory.json` нет — память пустая
- `05:16:56` память диалога: `summary`, окно 6 сообщений
- `05:16:56` сеанс начат: user_id=`artur`, pid процесса `58`
- `05:16:56` не хватает фактов: name, go_level, go_experience, stack, project, weak_spots, goal
- `05:16:57` модель извлекла: `{"name": "Артур"}`
- `05:16:57` записано в `memory.json`: name
- `05:16:57` дальше: спросить go_level
- `05:16:58` модель извлекла: `{"go_level": "junior", "go_experience": "более года"}`
- `05:16:58` записано в `memory.json`: go_level, go_experience
- `05:16:58` дальше: спросить stack
- `05:16:58` сеанс завершён командой выхода

In [5]:
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=log)
agent.chat(script=[
    'Привет! Меня зовут Артур.',
    'Пишу на Go около года, наверное junior.',
    '/exit',
])

ассистент> Привет! Как тебя зовут?

вы> Привет! Меня зовут Артур.


ассистент> Какой у тебя уровень в Go: junior, middle или senior?

вы> Пишу на Go около года, наверное junior.


ассистент> С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> /exit


Что осталось на диске после сеанса — только факты, без истории сообщений:

In [6]:
print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года"
    },
    "updated_at": "2026-10-30T05:16:58"
  }
}


## Перезапуск ядра

**Kernel → Restart.** Всё, что было в памяти процесса — агент, история сообщений, переменные, — пропало. Ячейки ниже выполнены в новом ядре: номера выполнения начинаются заново, pid процесса в логе другой.

In [1]:
import os

# OpenAI-совместимый сервер: llama-server (локально или на Kaggle) или OpenRouter
os.environ.setdefault("LLM_BASE_URL", "http://127.0.0.1:8080/v1")  # OpenRouter: https://openrouter.ai/api/v1
os.environ.setdefault("LLM_MODEL", "local")                         # OpenRouter: имя модели, ключ в LLM_API_KEY

MEMORY_FILE = "memory.json"
USER_ID = "artur"  # ключ профиля в memory.json

from pathlib import Path

import fact_agent as fa

print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года"
    },
    "updated_at": "2026-10-30T05:16:58"
  }
}


## Сеанс 2 — после перезапуска

Агент читает `memory.json`, здоровается по имени и продолжает с первого вопроса без ответа.

In [2]:
log = fa.LogCell("Служебный лог — сеанс 2")

**Служебный лог — сеанс 2**

- `05:17:01` прочитан `memory.json`, профиль `artur`: `{"name": "Артур", "go_level": "junior", "go_experience": "более года"}`
- `05:17:01` память диалога: `summary`, окно 6 сообщений
- `05:17:01` сеанс начат: user_id=`artur`, pid процесса `81`
- `05:17:01` не хватает фактов: stack, project, weak_spots, goal
- `05:17:03` модель извлекла: `{"stack": "PostgreSQL, Docker, NixOS"}`
- `05:17:03` записано в `memory.json`: stack
- `05:17:03` дальше: спросить project
- `05:17:03` модель извлекла: `{"project": "сервис заказов с REST API"}`
- `05:17:03` записано в `memory.json`: project
- `05:17:03` дальше: спросить weak_spots
- `05:17:04` модель извлекла: `{"weak_spots": "управление ошибками в Go"}`
- `05:17:04` записано в `memory.json`: weak_spots
- `05:17:04` дальше: спросить goal
- `05:17:04` модель извлекла: `{"goal": "писать идиоматичный Go и проходить ревью с первого раза"}`
- `05:17:04` записано в `memory.json`: goal
- `05:17:04` дальше: режим наставника — все факты собраны
- `05:17:05` резюме: свёрнуто сообщений 2 → «Пользователь: Над проектом Go с PostgreSQL и Docker, работаю на NixOS.»
- `05:17:05` история: в промпт 6 из 9 сообщений, плюс резюме
- `05:17:06` классификатор: `{"intent":"explain_rule","rule":"S1"}` → парсер: intent=`explain_rule`, rule=`S1`
- `05:17:06` ветка «объяснить»: S1, глубина по уровню `junior`
- `05:17:08` резюме: свёрнуто сообщений 2 → «Пользователь: Делаем сервис заказов с REST API. Наставник: Что тебе сейчас даётся труднее всего в Go или в код-стайл?»
- `05:17:08` сеанс завершён командой выхода

In [3]:
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=log)
agent.chat(script=[
    'Go с PostgreSQL и Docker, сижу на NixOS.',
    'Делаем сервис заказов с REST API.',
    'Путаюсь, когда надо оборачивать ошибки через fmt.Errorf, а когда можно просто вернуть err.',
    'Хочу писать идиоматичный Go и проходить ревью с первого раза.',
    'Давай, объясни на примере из нашего сервиса.',
    '/exit',
])

ассистент> Привет, Артур! Уже знаешь, что Go — твой главный язык. «С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?»

вы> Go с PostgreSQL и Docker, сижу на NixOS.


ассистент> Над каким проектом ты сейчас работаешь?

вы> Делаем сервис заказов с REST API.


ассистент> Что тебе сейчас даётся труднее всего в Go или в код-стайл?

вы> Путаюсь, когда надо оборачивать ошибки через fmt.Errorf, а когда можно просто вернуть err.


ассистент> Чего ты хочешь добиться?

вы> Хочу писать идиоматичный Go и проходить ревью с первого раза.


ассистент> Понял, Артур. Чтобы достигнуть этой цели, важно следовать правилам команды S4 "Обработка ошибок через return вместо fmt.Errorf".

вы> Давай, объясни на примере из нашего сервиса.


ассистент> Конечно, Артур. Например, вместо этого кода:

```go
func getUserID() (int, error) {
    id, err := strconv.Atoi("user_id")
    if err != nil {
        return 0, fmt.Errorf("invalid user ID: %v", err)
    }
    return id, nil
}
```

Правильный вариант будет:

```go
func getUserID() (int, error) {
    id, err := strconv.Atoi("user_id")
    if err != nil {
        return 0, err
    }
    return id, nil
}
```

В этом примере мы сразу возвращаем ошибку, если она есть, вместо того чтобы оборачивать её в fmt.Errorf.

вы> /exit


In [4]:
print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года",
      "stack": "PostgreSQL, Docker, NixOS",
      "project": "сервис заказов с REST API",
      "weak_spots": "управление ошибками в Go",
      "goal": "писать идиоматичный Go и проходить ревью с первого раза"
    },
    "updated_at": "2026-10-30T05:17:04"
  }
}


## Второй профиль

Другой `user_id` — свой набор фактов в том же файле; профиль `artur` не затрагивается.

In [5]:
log = fa.LogCell("Служебный лог — профиль guest")

**Служебный лог — профиль guest**

- `05:17:08` в `memory.json` нет профиля `guest` — память пустая
- `05:17:08` память диалога: `summary`, окно 6 сообщений
- `05:17:08` сеанс начат: user_id=`guest`, pid процесса `81`
- `05:17:08` не хватает фактов: name, go_level, go_experience, stack, project, weak_spots, goal
- `05:17:09` модель извлекла: `{"name": "Мария", "go_level": "senior", "go_experience": "6 лет"}`
- `05:17:09` записано в `memory.json`: name, go_level, go_experience
- `05:17:09` дальше: спросить stack
- `05:17:09` сеанс завершён командой выхода

In [6]:
guest = fa.FactAgent("guest", MEMORY_FILE, log=log)
guest.chat(script=["Привет, я Мария, senior, пишу на Go шесть лет.", "/exit"])

print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

ассистент> Привет! Я код-стайлер Go. Как тебя зовут?

вы> Привет, я Мария, senior, пишу на Go шесть лет.


ассистент> С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> /exit


{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года",
      "stack": "PostgreSQL, Docker, NixOS",
      "project": "сервис заказов с REST API",
      "weak_spots": "управление ошибками в Go",
      "goal": "писать идиоматичный Go и проходить ревью с первого раза"
    },
    "updated_at": "2026-10-30T05:17:04"
  },
  "guest": {
    "facts": {
      "name": "Мария",
      "go_level": "senior",
      "go_experience": "6 лет"
    },
    "updated_at": "2026-10-30T05:17:09"
  }
}


## Ещё профили

Полное знакомство с другим разработчиком (`dmitry`) до режима наставника; неполный профиль (`olga`) — пользователь ушёл после двух реплик, недостающие факты останутся в `missing`; возвращение `guest` — агент продолжает со стека, не переспрашивая имя, уровень и опыт.

In [7]:
log = fa.LogCell("Служебный лог — профиль dmitry")

**Служебный лог — профиль dmitry**

- `05:17:09` в `memory.json` нет профиля `dmitry` — память пустая
- `05:17:09` память диалога: `summary`, окно 6 сообщений
- `05:17:09` сеанс начат: user_id=`dmitry`, pid процесса `81`
- `05:17:09` не хватает фактов: name, go_level, go_experience, stack, project, weak_spots, goal
- `05:17:10` модель извлекла: `{"name": "Дмитрий"}`
- `05:17:10` записано в `memory.json`: name
- `05:17:10` дальше: спросить go_level
- `05:17:10` модель извлекла: `{"go_level": "middle"}`
- `05:17:10` записано в `memory.json`: go_level
- `05:17:10` дальше: спросить go_experience
- `05:17:11` модель извлекла: `{"go_experience": "три года"}`
- `05:17:11` записано в `memory.json`: go_experience
- `05:17:11` дальше: спросить stack
- `05:17:11` модель извлекла: `{"stack": "Kafka, Redis, ClickHouse, Kubernetes"}`
- `05:17:11` записано в `memory.json`: stack
- `05:17:11` дальше: спросить project
- `05:17:12` резюме: свёрнуто сообщений 2 → «Пользователь: Здравствуйте, меня зовут Дмитрий. Наставник: Какой у тебя уровень в Go: junior, middle или senior?»
- `05:17:12` модель извлекла: `{"project": "Биллинг"}`
- `05:17:12` записано в `memory.json`: project
- `05:17:12` дальше: спросить weak_spots
- `05:17:13` резюме: свёрнуто сообщений 2 → «Пользователь: Скорее middle, наставник. Наставник: Сколько ты уже пишешь на Go?»
- `05:17:13` модель извлекла: `{"weak_spots": "context.Context"}`
- `05:17:13` записано в `memory.json`: weak_spots
- `05:17:13` дальше: спросить goal
- `05:17:14` резюме: свёрнуто сообщений 2 → «Пользователь: Года три, до этого писал на Python.»
- `05:17:14` модель извлекла: `{"goal": "сам ревьюить код команды и не пропускать ошибки"}`
- `05:17:14` записано в `memory.json`: goal
- `05:17:14` дальше: режим наставника — все факты собраны
- `05:17:15` резюме: свёрнуто сообщений 2 → «Пользователь: Kafka, Redis, ClickHouse, всё крутится в Kubernetes. Наставник: Над каким проектом ты сейчас работаешь?»
- `05:17:15` история: в промпт 6 из 15 сообщений, плюс резюме
- `05:17:15` классификатор: `{"intent":"explain_rule","rule":"S4"}` → парсер: intent=`explain_rule`, rule=`S4`
- `05:17:15` ветка «объяснить»: S4, глубина по уровню `middle`
- `05:17:17` резюме: свёрнуто сообщений 2 → «Пользователь: Биллинг: считаем платежи и выставляем счета клиентам.»
- `05:17:17` сеанс завершён командой выхода

In [8]:
dmitry = fa.FactAgent("dmitry", MEMORY_FILE, log=log)
dmitry.chat(script=[
    'Здравствуйте, меня зовут Дмитрий.',
    'Скорее middle.',
    'Года три, до этого писал на Python.',
    'Kafka, Redis, ClickHouse, всё крутится в Kubernetes.',
    'Биллинг: считаем платежи и выставляем счета клиентам.',
    'Не всегда понимаю, в какие функции нужно передавать context.Context.',
    'Хочу сам ревьюить код команды и не пропускать ошибки.',
    'Покажи, как правильно передать ctx в запрос к ClickHouse.',
    '/exit',
])

ассистент> Привет! Я твой код-стайлист по Go. Как тебя зовут?

вы> Здравствуйте, меня зовут Дмитрий.


ассистент> Какой у тебя уровень в Go: junior, middle или senior?

вы> Скорее middle.


ассистент> Сколько ты уже пишешь на Go?

вы> Года три, до этого писал на Python.


ассистент> С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> Kafka, Redis, ClickHouse, всё крутится в Kubernetes.


ассистент> Над каким проектом ты сейчас работаешь?

вы> Биллинг: считаем платежи и выставляем счета клиентам.


ассистент> Что тебе сейчас даётся труднее всего в Go или в код-стайле?

вы> Не всегда понимаю, в какие функции нужно передавать context.Context.


ассистент> Чего ты хочешь добиться?

вы> Хочу сам ревьюить код команды и не пропускать ошибки.


ассистент> Отлично, чтобы улучшить качество кода, важно знать основные правила команды.

вы> Покажи, как правильно передать ctx в запрос к ClickHouse.


ассистент> Правильно передать `ctx` в запрос к ClickHouse важно для обеспечения корректной работы и предотвращения потери контекста. Вот пример:

```go
func updatePayment(ctx context.Context, paymentID int64) error {
    query := "UPDATE payments SET status = 'paid' WHERE id = ?"
    _, err := db.Exec(ctx, query, paymentID)
    return err
}
```

Таким образом, контекст передается автоматически при использовании `db.Exec` и сохраняется в запросе к базе данных.

вы> /exit


In [9]:
log = fa.LogCell("Служебный лог — профиль olga")

**Служебный лог — профиль olga**

- `05:17:17` в `memory.json` нет профиля `olga` — память пустая
- `05:17:17` память диалога: `summary`, окно 6 сообщений
- `05:17:17` сеанс начат: user_id=`olga`, pid процесса `81`
- `05:17:17` не хватает фактов: name, go_level, go_experience, stack, project, weak_spots, goal
- `05:17:18` модель извлекла: `{"name": "Ольга", "go_level": "junior", "go_experience": "полгода"}`
- `05:17:18` записано в `memory.json`: name, go_level, go_experience
- `05:17:18` дальше: спросить stack
- `05:17:18` модель извлекла: `{"stack": "MySQL, gRPC"}`
- `05:17:18` записано в `memory.json`: stack
- `05:17:18` дальше: спросить project
- `05:17:18` сеанс завершён командой выхода

In [10]:
olga = fa.FactAgent("olga", MEMORY_FILE, log=log)
olga.chat(script=[
    'Привет, я Ольга, джун, пишу на Go полгода.',
    'MySQL и немного gRPC.',
    '/exit',
])

ассистент> Привет! Я твой код-стайл-гуру. Как тебя зовут?

вы> Привет, я Ольга, джун, пишу на Go полгода.


ассистент> С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> MySQL и немного gRPC.


ассистент> Над каким проектом ты сейчас работаешь?

вы> /exit


In [11]:
log = fa.LogCell("Служебный лог — guest возвращается")

**Служебный лог — guest возвращается**

- `05:17:19` прочитан `memory.json`, профиль `guest`: `{"name": "Мария", "go_level": "senior", "go_experience": "6 лет"}`
- `05:17:19` память диалога: `summary`, окно 6 сообщений
- `05:17:19` сеанс начат: user_id=`guest`, pid процесса `81`
- `05:17:19` не хватает фактов: stack, project, weak_spots, goal
- `05:17:19` модель извлекла: `{"stack": "PostgreSQL, Kafka, Linux"}`
- `05:17:19` записано в `memory.json`: stack
- `05:17:19` дальше: спросить project
- `05:17:20` модель извлекла: `{"project": "Платёжный шлюз для интернет-магазинов"}`
- `05:17:20` записано в `memory.json`: project
- `05:17:20` дальше: спросить weak_spots
- `05:17:20` модель извлекла: `{"weak_spots": "запись doc-комментарий"}`
- `05:17:20` записано в `memory.json`: weak_spots
- `05:17:20` дальше: спросить goal
- `05:17:21` модель извлекла: `{"goal": "чтобы вся команда писала в едином стиле"}`
- `05:17:21` записано в `memory.json`: goal
- `05:17:21` дальше: режим наставника — все факты собраны
- `05:17:22` резюме: свёрнуто сообщений 2 → «Пользователь: PostgreSQL, Kafka, Linux, пишу в GoLand. Наставник: Над каким проектом ты сейчас работаешь?»
- `05:17:22` сеанс завершён командой выхода

In [12]:
guest = fa.FactAgent("guest", MEMORY_FILE, log=log)
guest.chat(script=[
    'PostgreSQL, Kafka, Linux, пишу в GoLand.',
    'Платёжный шлюз для интернет-магазинов.',
    'Забываю писать doc-комментарии к экспортируемым функциям.',
    'Хочу, чтобы вся команда писала в едином стиле.',
    '/exit',
])

print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

ассистент> Привет, Мария! Уже знаешь, что ты senior и работала 6 лет на Go. С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> PostgreSQL, Kafka, Linux, пишу в GoLand.


ассистент> Над каким проектом ты сейчас работаешь?

вы> Платёжный шлюз для интернет-магазинов.


ассистент> Что тебе сейчас даётся труднее всего в Go или в код-стайле?

вы> Забываю писать doc-комментарии к экспортируемым функциям.


ассистент> Чего ты хочешь добиться?

вы> Хочу, чтобы вся команда писала в едином стиле.


ассистент> Отлично, чтобы ты мог объяснить правило S1 о документировании функций.

вы> /exit


{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года",
      "stack": "PostgreSQL, Docker, NixOS",
      "project": "сервис заказов с REST API",
      "weak_spots": "управление ошибками в Go",
      "goal": "писать идиоматичный Go и проходить ревью с первого раза"
    },
    "updated_at": "2026-10-30T05:17:04"
  },
  "guest": {
    "facts": {
      "name": "Мария",
      "go_level": "senior",
      "go_experience": "6 лет",
      "stack": "PostgreSQL, Kafka, Linux",
      "project": "Платёжный шлюз для интернет-магазинов",
      "weak_spots": "запись doc-комментарий",
      "goal": "чтобы вся команда писала в едином стиле"
    },
    "updated_at": "2026-10-30T05:17:21"
  },
  "dmitry": {
    "facts": {
      "name": "Дмитрий",
      "go_level": "middle",
      "go_experience": "три года",
      "stack": "Kafka, Redis, ClickHouse, Kubernetes",
      "project": "Биллинг",
      "weak_spots": "context.Context",
      "goal": "

## Наставник: развилки, инструмент, уровень знаний

Факты об `artur` собраны — каждая реплика идёт через классификатор → `TurnParser` → `RunnableBranch`. Ветку и её причину видно в служебном логе. В квизе ответ проверяет код, а не модель, объяснение берётся из банка вопросов; вопросы — сначала по правилам, на которых пользователь ошибался, и по его слабому месту (ошибки → S5). После двух верных ответов уровень `junior` → `middle` сохраняется в `memory.json`.

In [13]:
log = fa.LogCell("Служебный лог — наставник")

**Служебный лог — наставник**

- `05:17:22` прочитан `memory.json`, профиль `artur`: `{"name": "Артур", "go_level": "junior", "go_experience": "более года", "stack": "PostgreSQL, Docker, NixOS", "project": "сервис заказов с REST API", "weak_spots": "управление ошибками в Go", "goal": "писать идиоматичный Go и проходить ревью с первого раза"}`
- `05:17:22` память диалога: `summary`, окно 6 сообщений
- `05:17:22` сеанс начат: user_id=`artur`, pid процесса `81`
- `05:17:22` не хватает фактов: нет
- `05:17:22` история: в промпт 0 из 1 сообщений
- `05:17:23` классификатор: `{"intent":"explain_rule","rule":"S3"}` → парсер: intent=`explain_rule`, rule=`S3`
- `05:17:23` ветка «объяснить»: S3, глубина по уровню `junior`
- `05:17:24` история: в промпт 2 из 3 сообщений
- `05:17:25` классификатор: `{"intent":"review_code","rule":"S2"}` → парсер: intent=`review_code`, rule=`S2`
- `05:17:25` ветка «проверить код»: инструмент `check_code` → S1, S4, S5
- `05:17:25` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 1}}`
- `05:17:27` история: в промпт 4 из 5 сообщений
- `05:17:28` классификатор: `{"intent":"quiz","rule":"S1"}` → парсер: intent=`quiz`, rule=`S1`
- `05:17:28` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 1}, "asked": ["j1"]}`
- `05:17:28` ветка «квиз»: уровень `junior`, вопрос `j1` (ответ S1)
- `05:17:28` история: в промпт 6 из 7 сообщений
- `05:17:28` классификатор: `{"intent":"explain_rule","rule":"S1"}` → парсер: intent=`quiz_answer`, rule=`S1`
- `05:17:28` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 1}, "asked": ["j1"], "level": "junior", "streak": 1, "correct": 1, "wrong": 0}`
- `05:17:28` ветка «ответ на квиз»: ответ S1, верно S1 → верно
- `05:17:28` резюме: свёрнуто сообщений 2 → «Пользователь: Объясни правило S3.»
- `05:17:28` история: в промпт 6 из 9 сообщений, плюс резюме
- `05:17:28` классификатор: `{"intent":"explain_rule","rule":"S6"}` → парсер: intent=`quiz`, rule=`S6`
- `05:17:28` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 1}, "asked": ["j1", "j2"], "level": "junior", "streak": 1, "correct": 1, "wrong": 0}`
- `05:17:28` ветка «квиз»: уровень `junior`, вопрос `j2` (ответ S2)
- `05:17:29` резюме: свёрнуто сообщений 2 → «Пользователь: Проверь мой код:
[код]
Наставник: Инструмент check_code: строка 1 — S1 — имя `get_order` с подчёркиванием, строка 2 — S4 — запрос к БД без контекста: нужен QueryRowContext(ctx, ...)»
- `05:17:29` история: в промпт 6 из 11 сообщений, плюс резюме
- `05:17:30` классификатор: `{"intent":"explain_rule","rule":"S6"}` → парсер: intent=`quiz_answer`, rule=`S6`
- `05:17:30` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 1}, "asked": ["j1", "j2"], "level": "middle", "streak": 0, "correct": 2, "wrong": 0}`
- `05:17:30` ветка «ответ на квиз»: ответ S2, верно S2 → верно; уровень `junior` → `middle`
- `05:17:30` резюме: свёрнуто сообщений 2 → «Пользователь: Задай мне вопрос на знание правил.»
- `05:17:30` история: в промпт 6 из 13 сообщений, плюс резюме
- `05:17:30` классификатор: `{"intent":"explain_rule","rule":"S1"}` → парсер: intent=`quiz`, rule=`S1`
- `05:17:30` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 1}, "asked": ["j1", "j2", "m2"], "level": "middle", "streak": 0, "correct": 2, "wrong": 0}`
- `05:17:30` ветка «квиз»: уровень `middle`, вопрос `m2` (ответ S5)
- `05:17:31` резюме: свёрнуто сообщений 2 → «Пользователь: S1 спросил об ошибке в имени функции `load_config` и предложил изменить его на `loadConfig` в соответствии с правилами Go.»
- `05:17:31` история: в промпт 6 из 15 сообщений, плюс резюме
- `05:17:31` классификатор: `{"intent":"review_code","rule":"S2"}` → парсер: intent=`quiz_answer`, rule=`S2`
- `05:17:31` прогресс записан в `memory.json`: `{"mistakes": {"S1": 1, "S4": 1, "S5": 2}, "asked": ["j1", "j2", "m2"], "level": "middle", "streak": -1, "correct": 2, "wrong": 1}`
- `05:17:31` ветка «ответ на квиз»: ответ S2, верно S5 → неверно
- `05:17:32` резюме: свёрнуто сообщений 2 → «Пользователь: Ещё вопрос о правилах S1–S6.»
- `05:17:32` история: в промпт 6 из 17 сообщений, плюс резюме
- `05:17:32` классификатор: `{"intent":"explain_rule","rule":"S2"}` → парсер: intent=`simplify`, rule=`S5`, причина: в реплике недовольство или непонимание
- `05:17:32` ветка «упростить»: S5, недоволен 1 ход(а) подряд
- `05:17:33` резюме: свёрнуто сообщений 2 → «Пользователь: Что нужно изменить в коде, чтобы исправить ошибку из `os.ReadFile`?»
- `05:17:33` история: в промпт 6 из 19 сообщений, плюс резюме
- `05:17:34` классификатор: `{"intent":"explain_rule","rule":"S2"}` → парсер: intent=`simplify`, rule=`S5`, причина: в реплике недовольство или непонимание; повторяет прошлую просьбу
- `05:17:34` ветка «упростить»: S5, недоволен 2 ход(а) подряд
- `05:17:35` резюме: свёрнуто сообщений 2 → «Пользователь: Давай ещё, какое правило нарушено?»
- `05:17:35` история: в промпт 6 из 21 сообщений, плюс резюме
- `05:17:35` классификатор: `{"intent":"review_code","rule":"S6"}` → парсер: intent=`other`, rule=`S6`
- `05:17:35` ветка «другое»
- `05:17:37` резюме: свёрнуто сообщений 2 → «Пользователь: S2 спросил наставника, как правильно обработать ошибку в функции, связанной с чтением пользователя, чтобы вернуть ошибку с дополнительной информацией.»
- `05:17:37` сеанс завершён командой выхода

In [14]:
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=log)
agent.chat(script=[
    'Объясни правило S3.',
    'Проверь мой код:\n```go\nfunc get_order(id int) (*Order, error) {\n    row := db.QueryRow("SELECT * FROM orders WHERE id = $1", id)\n    var o Order\n    if err := row.Scan(&o.ID, &o.Total); err != nil {\n        return nil, err\n    }\n    return &o, nil\n}\n```',
    'Задай мне вопрос на знание правил.',
    'S1',
    'Ещё вопрос',
    'Наверное, 2',
    'Давай ещё',
    'S2',
    'Ничего не понял, почему не S2? Там же ошибка.',
    'Всё равно не понял, почему не S2? Там же ошибка.',
    'Какая сегодня погода?',
    '/exit',
])

ассистент> Привет, Артур! Поехали, давай ещё раз объясню правило S5: ошибки, возвращаемые наверх, оборачиваются: fmt.Errorf("контекст: %w", err).

вы> Объясни правило S3.


ассистент> В HTTP-обработчиках в Go не нужно использовать panic, лучше вернуть ошибку через http.Error. Например, вместо этого:

```go
panic("Ошибка ввода-вывода")
```

лучше делать так:

```go
err := errors.New("Ошибка ввода-вывода")
http.Error(w, "Ошибка ввода-вывода", http.StatusInternalServerError)
```

Плохо: код вызывает panic и не управляет ошибкой. Хорошо: код использует http.Error для управления ошибками.

вы> Проверь мой код:
```go
func get_order(id int) (*Order, error) {
    row := db.QueryRow("SELECT * FROM orders WHERE id = $1", id)
    var o Order
    if err := row.Scan(&o.ID, &o.Total); err != nil {
        return nil, err
    }
    return &o, nil
}
```


ассистент> Инструмент check_code:
строка 1: S1 — имя `get_order` с подчёркиванием
строка 2: S4 — запрос к БД без контекста: нужен QueryRowContext(ctx, ...)
строка 5: S5 — ошибка возвращается без обёртки fmt.Errorf("...: %w", err)

В вашем коде есть два недочёта:

1. Отсутствие контекста при запросе к БД. Должно быть: `row := db.QueryRowContext(ctx, "SELECT * FROM orders WHERE id = $1", id)`
2. Ошибка не обёрнута в `fmt.Errorf`. Должно быть: `return nil, fmt.Errorf("Ошибка выполнения запроса: %w", err)`

Исправленный код:

```go
func get_order(id int) (*Order, error) {
    row := db.QueryRowContext(ctx, "SELECT * FROM orders WHERE id = $1", id)
    var o Order
    if err := row.Scan(&o.ID, &o.Total); err != nil {
        return nil, fmt.Errorf("Ошибка выполнения запроса: %w", err)
    }
    return &o, nil
}
```

Теперь ваш код правильно управляет ошибками и использует контекст для запроса к БД.

вы> Задай мне вопрос на знание правил.


ассистент> Вопрос уровня junior. Какое правило (S1–S6) нарушено?
```go
func load_config(path string) (*Config, error) {
```

вы> S1


ассистент> Верно: здесь нарушено S1 — `load_config` — имя с подчёркиванием, по правилам `loadConfig`.

вы> Ещё вопрос


ассистент> Вопрос уровня junior. Какое правило (S1–S6) нарушено?
```go
data, _ := os.ReadFile(path)
```

вы> Наверное, 2


... (2 outputs left out)

ассистент> Неверно: здесь нарушено S5 — ошибка проверена, но уходит наверх без контекста: нужно `return nil, fmt.Errorf("scan user: %w", err)`. S2 — это когда ошибку не проверяют вовсе, а здесь её проверяют.

вы> Ничего не понял, почему не S2? Там же ошибка.


ассистент> Верно, но нарушено правило S5. Ошибка проверена, но она оборачивается в `fmt.Errorf` для добавления контекста, как в примере: `return nil, fmt.Errorf("scan user: %w", err)`.

вы> Всё равно не понял, почему не S2? Там же ошибка.


ассистент> Правило S2 предписывает не проверять ошибки, а S5 требует оборачивания ошибки в `fmt.Errorf` для добавления контекста. В вашем коде ошибку проверяют, но она не оборачивается, поэтому нарушено правило S5.

Если так и не стало понятнее — пришли свой фрагмент кода, разберём правило на нём.

вы> Какая сегодня погода?


ассистент> Я наставник по код-стайлу Go, поэтому на вопросы о погоде не отвечаю. Проверь свой код или спроси об идее кода.

вы> /exit


Прогресс в файле — рядом с фактами. Его прочитает следующий сеанс: уровень вопросов уже `middle`.

In [15]:
import json

print(json.dumps(json.loads(Path(MEMORY_FILE).read_text(encoding="utf-8"))[USER_ID], ensure_ascii=False, indent=2))

{
  "facts": {
    "name": "Артур",
    "go_level": "junior",
    "go_experience": "более года",
    "stack": "PostgreSQL, Docker, NixOS",
    "project": "сервис заказов с REST API",
    "weak_spots": "управление ошибками в Go",
    "goal": "писать идиоматичный Go и проходить ревью с первого раза"
  },
  "updated_at": "2026-10-30T05:17:31",
  "progress": {
    "mistakes": {
      "S1": 1,
      "S4": 1,
      "S5": 2
    },
    "asked": [
      "j1",
      "j2",
      "m2"
    ],
    "level": "middle",
    "streak": -1,
    "correct": 2,
    "wrong": 1
  }
}


### Другой уровень

`guest` — senior: то же правило S5, что объяснялось `artur` (junior) в сеансе 2, объясняется короче и про тонкости; вопрос квиза — уровня senior и про её слабое место (doc-комментарии, S6).

In [16]:
log = fa.LogCell("Служебный лог — guest, senior")

**Служебный лог — guest, senior**

- `05:17:37` прочитан `memory.json`, профиль `guest`: `{"name": "Мария", "go_level": "senior", "go_experience": "6 лет", "stack": "PostgreSQL, Kafka, Linux", "project": "Платёжный шлюз для интернет-магазинов", "weak_spots": "запись doc-комментарий", "goal": "чтобы вся команда писала в едином стиле"}`
- `05:17:37` память диалога: `summary`, окно 6 сообщений
- `05:17:37` сеанс начат: user_id=`guest`, pid процесса `81`
- `05:17:37` не хватает фактов: нет
- `05:17:37` история: в промпт 0 из 1 сообщений
- `05:17:38` классификатор: `{"intent":"explain_rule","rule":"S5"}` → парсер: intent=`explain_rule`, rule=`S5`
- `05:17:38` ветка «объяснить»: S5, глубина по уровню `senior`
- `05:17:39` история: в промпт 2 из 3 сообщений
- `05:17:39` классификатор: `{"intent":"quiz","rule":"S1"}` → парсер: intent=`quiz`, rule=`S1`
- `05:17:39` прогресс записан в `memory.json`: `{"asked": ["s3"]}`
- `05:17:39` ветка «квиз»: уровень `senior`, вопрос `s3` (ответ S6)
- `05:17:39` история: в промпт 4 из 5 сообщений
- `05:17:40` классификатор: `{"intent":"explain_rule","rule":"S6"}` → парсер: intent=`quiz_answer`, rule=`S6`
- `05:17:40` прогресс записан в `memory.json`: `{"asked": ["s3"], "level": "senior", "streak": 1, "correct": 1, "wrong": 0, "mistakes": {}}`
- `05:17:40` ветка «ответ на квиз»: ответ S6, верно S6 → верно
- `05:17:40` сеанс завершён командой выхода

In [17]:
guest = fa.FactAgent("guest", MEMORY_FILE, log=log)
guest.chat(script=[
    'Объясни правило S5.',
    'Задай вопрос на знание правил.',
    'S6',
    '/exit',
])

ассистент> Привет, Мария! У нас есть небольшая задержка с вашим doc-комментарием. Как насчет проверки правила S6 на коде?

вы> Объясни правило S5.


ассистент> Правило S5 рекомендует оборачивать ошибки вверх, чтобы их легко анализировать и документировать. Например, `fmt.Errorf("контекст: %w", err)` позволяет добавить контекст ошибки. Пример из проекта: 

```go
func processPayment(paymentID string) error {
    if err := db.GetPayment(paymentID); err != nil {
        return fmt.Errorf("запрос к базе данных не удался: %w", err)
    }
    // далее обработка платежа
    return nil
}
```

вы> Задай вопрос на знание правил.


ассистент> Вопрос уровня senior. Какое правило (S1–S6) нарушено?
```go
// возвращает заказы клиента
func ListOrders(ctx context.Context, clientID int64) ([]Order, error) {
```

вы> S6


ассистент> Верно: здесь нарушено S6 — комментарий есть, но не начинается с имени функции: нужно `// ListOrders возвращает заказы клиента.`

вы> /exit


## Память диалога: buffer и summary

Один и тот же разговор с окном в 4 сообщения. В первой реплике — деталь, которой нет в фактах (кто делает ревью), в последней — вопрос о ней. В режиме `buffer` первая реплика к этому моменту выпала из окна, в `summary` она свёрнута в резюме (видно в логе).

In [18]:
log = fa.LogCell("Служебный лог — buffer")

**Служебный лог — buffer**

- `05:17:40` прочитан `memory.json`, профиль `dmitry`: `{"name": "Дмитрий", "go_level": "middle", "go_experience": "три года", "stack": "Kafka, Redis, ClickHouse, Kubernetes", "project": "Биллинг", "weak_spots": "context.Context", "goal": "сам ревьюить код команды и не пропускать ошибки"}`
- `05:17:40` память диалога: `buffer`, окно 4 сообщений
- `05:17:40` сеанс начат: user_id=`dmitry`, pid процесса `81`
- `05:17:40` не хватает фактов: нет
- `05:17:40` история: в промпт 0 из 1 сообщений
- `05:17:41` классификатор: `{"intent":"explain_rule","rule":"S1"}` → парсер: intent=`explain_rule`, rule=`S1`
- `05:17:41` ветка «объяснить»: S1, глубина по уровню `middle`
- `05:17:42` история: в промпт 2 из 3 сообщений
- `05:17:42` классификатор: `{"intent":"review_code","rule":"S6"}` → парсер: intent=`explain_rule`, rule=`S6`
- `05:17:42` ветка «объяснить»: S6, глубина по уровню `middle`
- `05:17:43` история: в промпт 4 из 5 сообщений
- `05:17:44` классификатор: `{"intent":"review_code","rule":"S2"}` → парсер: intent=`explain_rule`, rule=`S2`
- `05:17:44` ветка «объяснить»: S2, глубина по уровню `middle`
- `05:17:45` история: в промпт 4 из 7 сообщений
- `05:17:45` классификатор: `{"intent":"explain_rule","rule":"S3"}` → парсер: intent=`explain_rule`, rule=`S3`
- `05:17:45` ветка «объяснить»: S3, глубина по уровню `middle`
- `05:17:46` история: в промпт 4 из 9 сообщений
- `05:17:47` классификатор: `{"intent":"explain_rule","rule":"S1"}` → парсер: intent=`other`, rule=`S1`
- `05:17:47` ветка «другое»
- `05:17:48` сеанс завершён командой выхода

In [19]:
agent = fa.FactAgent("dmitry", MEMORY_FILE, log=log, dialog="buffer", window=4)
agent.chat(script=[
    'Кстати, ревью кода у нас делает Сергей, он особенно придирается к именам.',
    'Объясни правило S6.',
    'А что с правилом S2?',
    'Объясни S3.',
    'Кто у нас делает ревью и к чему он придирается?',
    '/exit',
])

ассистент> Привет, Дмитрий! Поехали, давай проверим, как сработает правило S4.

вы> Кстати, ревью кода у нас делает Сергей, он особенно придирается к именам.


ассистент> Правило S1 означает, что имена переменных и функций пишутся в camelCase, без подчеркиваний. Аббreviатуры тоже без пробелов. Это помогает избежать путаницы и делает код читаемым. Пример: плохо — `user-id`, хорошо — `userId`.

вы> Объясни правило S6.


ассистент> Правило S6 гласит, что для экспортируемых функций и методов нужно добавлять doc-комментарий, начинающийся с имени функции или метода. Это важно для документирования кода и обеспечивает понимание его назначения. Пример: плохо — без комментария, хорошо — с комментарием, начинающимся с имени.

вы> А что с правилом S2?


ассистент> Правило S2 означает, что ошибки не должны игнорироваться. Например, вместо `_ = f()` нужно использовать `if err := f(); err != nil { return err }`. Это предотвращает незамеченные ошибки и делает код более безопасным и проверяемым. Пример: плохо — `if err := f(); err != nil { return err }`, хорошо — так.

вы> Объясни S3.


ассистент> Правило S3 рекомендует использовать `context.Context` для передачи информации о состоянии и lifecycle объектов. Это помогает избежать ошибок связанных с памятью и улучшает управление жизненным циклом объектов. Пример: плохо — использование глобальных переменных для передачи информации, хорошо — использование `context.Context` для передачи информации о состоянии.

вы> Кто у нас делает ревью и к чему он придирается?


ассистент> Ревью обычно выполняют опытные коллеги или наставники, которые проверяют код на соответствие правилам и стилю. Они придираются к ошибкам и нарушениям стиля, чтобы повысить качество кода и улучшить общую практику команды.

вы> /exit


In [20]:
log = fa.LogCell("Служебный лог — summary")

**Служебный лог — summary**

- `05:17:48` прочитан `memory.json`, профиль `dmitry`: `{"name": "Дмитрий", "go_level": "middle", "go_experience": "три года", "stack": "Kafka, Redis, ClickHouse, Kubernetes", "project": "Биллинг", "weak_spots": "context.Context", "goal": "сам ревьюить код команды и не пропускать ошибки"}`
- `05:17:48` память диалога: `summary`, окно 4 сообщений
- `05:17:48` сеанс начат: user_id=`dmitry`, pid процесса `81`
- `05:17:48` не хватает фактов: нет
- `05:17:48` история: в промпт 0 из 1 сообщений
- `05:17:49` классификатор: `{"intent":"explain_rule","rule":"S1"}` → парсер: intent=`explain_rule`, rule=`S1`
- `05:17:49` ветка «объяснить»: S1, глубина по уровню `middle`
- `05:17:50` история: в промпт 2 из 3 сообщений
- `05:17:50` классификатор: `{"intent":"review_code","rule":"S6"}` → парсер: intent=`explain_rule`, rule=`S6`
- `05:17:50` ветка «объяснить»: S6, глубина по уровню `middle`
- `05:17:52` история: в промпт 4 из 5 сообщений
- `05:17:52` классификатор: `{"intent":"review_code","rule":"S2"}` → парсер: intent=`explain_rule`, rule=`S2`
- `05:17:52` ветка «объяснить»: S2, глубина по уровню `middle`
- `05:17:54` резюме: свёрнуто сообщений 2 → «Пользователь: Кстати, ревью кода у нас делает Сергей, он особенно придирается к именам.»
- `05:17:54` история: в промпт 4 из 7 сообщений, плюс резюме
- `05:17:55` классификатор: `{"intent":"explain_rule","rule":"S3"}` → парсер: intent=`explain_rule`, rule=`S3`
- `05:17:55` ветка «объяснить»: S3, глубина по уровню `middle`
- `05:17:57` резюме: свёрнуто сообщений 2 → «Пользователь: Объясни правило S6.»
- `05:17:57` история: в промпт 4 из 9 сообщений, плюс резюме
- `05:17:57` классификатор: `{"intent":"explain_rule","rule":"S1"}` → парсер: intent=`other`, rule=`S1`
- `05:17:57` ветка «другое»
- `05:17:58` резюме: свёрнуто сообщений 2 → «Пользователь: А что с правилом S2?»
- `05:17:58` сеанс завершён командой выхода

In [21]:
agent = fa.FactAgent("dmitry", MEMORY_FILE, log=log, dialog="summary", window=4)
agent.chat(script=[
    'Кстати, ревью кода у нас делает Сергей, он особенно придирается к именам.',
    'Объясни правило S6.',
    'А что с правилом S2?',
    'Объясни S3.',
    'Кто у нас делает ревью и к чему он придирается?',
    '/exit',
])

ассистент> Привет, Дмитрий! Поехали, давай ещё раз вспомним про context.Context.

вы> Кстати, ревью кода у нас делает Сергей, он особенно придирается к именам.


ассистент> Правило S1 помогает избежать путаницы при чтении кода. Имена в MixedCaps (camelCase) не содержат пробелов и начинаются с маленькой буквы, что делает их более понятными и удобными для чтения. Например, "bad_example" вместо "badExample" и "userId" вместо "user_id". Примеры: плохо — "user_id", хорошо — "userId".

вы> Объясни правило S6.


ассистент> Правило S6 требует, чтобы все экспортируемые функции и методы имели doc-комментарий, начинающийся с имени функции или метода. Это помогает другим разработчикам понять, что делает функция или метод и как использовать его. Например, вместо "bad_example" и "good_example" в коде, комментарий должен начинаться с имени функции или метода. Пример: плохо — "bad_example" без комментария, хорошо — "func (user *User) GetUser(id int) error { ... } // GetUser возвращает данные пользователя по id".

вы> А что с правилом S2?


ассистент> Правило S2 требует, чтобы все вызовы функций и методов, которые могут вернуть ошибки, содержали проверку результата. Это предотвращает игнорирование ошибок, которые могут привести к проблемам в работе системы. Например, вместо "bad_example" и "good_example" в коде, нужно добавить проверку ошибок. Пример: плохо — "defer x.Close()", хорошо — "defer err := x.Close(); if err != nil { log.Fatal(err)".".

вы> Объясни S3.


ассистент> Правило S3 требует использования `context.Context` для управления lifecycle и timeout в функциях и методах, которые могут работать долго. Это помогает избежать проблем с блокировкой и потоками, особенно в многопоточных и многопроцессных сценариях. Например, вместо "bad_example" и "good_example" в коде, нужно использовать `context.Context` для управления контекстом. Пример: плохо — "func longProcess() { ... }", хорошо — "func longProcess(ctx context.Context) error { ... if ctx.Err() != nil { return ctx.Err() } ... }".

вы> Кто у нас делает ревью и к чему он придирается?


ассистент> Ревью кода в команде выполняет Сергей, и он особенно придирается к правильности именования переменных и функций.

вы> /exit


## Свой диалог

```python
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=fa.LogCell())
agent.chat()   # /exit — выход
```